# LLaVA-1.5-7B answers with gaze-foveated patching — your own images (no training)

Same as `llava_foveated.ipynb`, but on images you upload from your computer: for each image, normal LLaVA (576 image tokens) vs gaze-foveated LLaVA (14 px fovea / 28 px ring / 56 px periphery through APT) at three gaze points (centre, corner, right), on two prompts. Then shows the layouts the model was given.

Images are resized and centre-cropped to 336×336 (LLaVA's input), so gaze points refer to that square crop.

**Runtime:** GPU. T4 works (LLM in 4-bit, vision encoder fp16); L4/A100 runs the LLM in fp16. LLaVA weights (~14 GB) download on first run.

In [ ]:
import os

if not os.path.exists('apt-vlm'):
    !git clone -q https://github.com/shubhamOjha1000/apt-vlm.git
else:
    !git -C apt-vlm pull -q
!git -C apt-vlm log --oneline -1

In [ ]:
!pip install -q -U timm einops ipdb bitsandbytes accelerate pillow-heif
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Upload images from your computer (jpg, png, webp, bmp, heic). Re-running this cell replaces the previous set.
import shutil
from google.colab import files

UPLOAD_DIR = 'apt-vlm/uploads'
shutil.rmtree(UPLOAD_DIR, ignore_errors=True)
os.makedirs(UPLOAD_DIR)

uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join(UPLOAD_DIR, name), 'wb') as f:
        f.write(data)
print(f'{len(uploaded)} image(s) in {UPLOAD_DIR}:', sorted(os.listdir(UPLOAD_DIR)))

In [ ]:
# Options: --a 40 (fall-off), --fov 110 (field of view, degrees), --num-scales 3, --max-new-tokens 60
!rm -rf apt-vlm/outputs/llava_foveated
!cd apt-vlm && APT_ATTN_IMPL=eager python scripts/llava_foveated.py --image-dir uploads

In [ ]:
import glob
from IPython.display import Image, display

for path in sorted(glob.glob('apt-vlm/outputs/llava_foveated/*.png')):
    print(os.path.basename(path), '— panels: gaze centre, corner, right')
    display(Image(filename=path))